# 85 — Creative Mega-Ensemble

Combines all creative OOF predictions (nb76–nb84) with the existing best models
(nb36 grand ensemble v6b, nb35 Chemprop auxiliary, nb42 SMOTE-ADASYN etc.)
using ElasticNetCV stacking.

Also applies the Delta-ML correction on top of the ensemble:
final = ensemble_pred + alpha * (delta_correction for high-similarity test compounds)


In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import stats
from pathlib import Path
from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, morgan_fp_batch, standardize_smiles, compute_physchem
from pxr.paths import DATA_PROCESSED, DATA_EXTERNAL, SUBMISSIONS
SEED = 42; N_FOLDS = 5
LGBM = dict(n_estimators=1000, num_leaves=64, learning_rate=0.05,
            min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)


In [2]:
def full_metrics(y_true, y_pred, cp=None, label=""):
    yt = np.asarray(y_true, float); yp = np.asarray(y_pred, float)
    msk = np.isfinite(yt) & np.isfinite(yp); yt, yp = yt[msk], yp[msk]
    mae = float(np.mean(np.abs(yt-yp)))
    rae_v = mae / float(np.mean(np.abs(yt-yt.mean()))) if yt.std()>0 else float("nan")
    r2  = 1-np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2) if yt.std()>0 else float("nan")
    pr, _ = stats.pearsonr(yt, yp); sp, _ = stats.spearmanr(yt, yp)
    kt, _ = stats.kendalltau(yt, yp)
    m = dict(RAE=rae_v, MAE=mae, R2=float(r2), Pearson=float(pr),
             Spearman=float(sp), Kendall=float(kt))
    if cp is not None and hasattr(cp, "iterrows") and len(cp) > 0:
        c=t=0
        for _,row in cp.iterrows():
            ia,ii = int(row.get("idx_active",-1)), int(row.get("idx_inactive",-1))
            if 0<=ia<len(yp) and 0<=ii<len(yp): c+=int(yp[ia]>yp[ii]); t+=1
        m["Cliff_acc"] = c/t if t else float("nan")
    if label:
        ca = f"  Cliff={m.get('Cliff_acc',float('nan')):.3f}" if "Cliff_acc" in m else ""
        print(f"  [{label}] RAE={rae_v:.4f} MAE={mae:.4f} R²={r2:.4f} "
              f"r={pr:.4f} ρ={sp:.4f} τ={kt:.4f}{ca}")
    return m

In [3]:
tr = load_train(); te = load_test()
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, N_FOLDS, SEED)
active_mask = y_tr >= 5.5
X_tr = impute(combined(tr["smiles"].tolist()))
X_te = impute(combined(te["smiles"].tolist()))
fps_tr = morgan_fp_batch(tr["smiles"].tolist()).astype(np.float32)
fps_te = morgan_fp_batch(te["smiles"].tolist()).astype(np.float32)
cliff_pairs = (pd.read_parquet(DATA_PROCESSED/"cliff_pairs.parquet")
               if (DATA_PROCESSED/"cliff_pairs.parquet").exists() else pd.DataFrame())
print(f"Train {len(tr):,}  Test {len(te):,}  Cliffs {len(cliff_pairs)}")


Train 4,139  Test 513  Cliffs 149


In [4]:
from sklearn.linear_model import ElasticNetCV
from pathlib import Path

# Discover all OOF predictions
oof_files = sorted(DATA_PROCESSED.glob("oof_*.npy"))
print(f"Discovered {len(oof_files)} OOF files")

oofs, tes, names = [], [], []
for fp in oof_files:
    name = fp.stem.replace("oof_","")
    te_fp = DATA_PROCESSED / f"te_{fp.stem.replace('oof_','')}.npy"
    try:
        oof = np.load(fp)
        if len(oof) != len(y_tr): continue
        te_v = np.load(te_fp) if te_fp.exists() else None
        if te_v is None or len(te_v) != 513: continue
        if not np.isfinite(oof).all(): oof[~np.isfinite(oof)] = y_tr.mean()
        if not np.isfinite(te_v).all(): te_v[~np.isfinite(te_v)] = float(np.nanmean(te_v))
        oofs.append(oof); tes.append(te_v); names.append(name)
    except Exception as e:
        print(f"  skip {name}: {e}")

print(f"Using {len(names)} models: {names}")
OOF_stack = np.column_stack(oofs)
TE_stack  = np.column_stack(tes)


Discovered 72 OOF files


Using 36 models: ['aux_features', 'bert_smiles', 'catboost', 'chemberta', 'chemberta_esm2_nr', 'chemberta_mtr', 'chemprop_aux', 'crossattn_chemberta_esm2', 'crossattn_grover_esm2', 'deep_ensemble', 'grand15', 'grand18', 'grand23', 'grand24', 'grand25', 'grand_v6', 'grand_v6b', 'grand_v6c', 'grover', 'grover_large', 'hard_negatives', 'knn', 'lgbm_all_external', 'lgbm_aug', 'lgbm_base', 'lgbm_bindingdb', 'lgbm_cliff_oversample', 'lgbm_pubchem', 'lgbm_tuned', 'morgan_esm2_nr', 'morgan_protbert_nr', 'nr_weighted', 'selformer', 'singleconc', 'unimol', 'xgboost_dart']


In [5]:
# ElasticNet meta-learner
meta = ElasticNetCV(l1_ratio=[0.1, 0.5, 0.9, 1.0], cv=5, max_iter=10000,
                     random_state=SEED)
meta.fit(OOF_stack, y_tr)
oof = meta.predict(OOF_stack)

coef_df = pd.DataFrame({"model": names, "weight": meta.coef_}).sort_values("weight", ascending=False)
print("Non-zero model weights:")
print(coef_df[coef_df.weight != 0].to_string(index=False))

m_ens = full_metrics(y_tr, oof, cliff_pairs, "mega_ensemble")
m_ens_a = full_metrics(y_tr[active_mask], oof[active_mask], "mega_ensemble [active]")
print("\n" + pd.DataFrame([m_ens, m_ens_a], index=["overall","active"]).round(4).to_string())


Non-zero model weights:
        model   weight
 aux_features 0.987405
deep_ensemble 0.017325
 chemprop_aux 0.006648
   lgbm_tuned 0.004815
     grand_v6 0.000713
  [mega_ensemble] RAE=0.2181 MAE=0.1985 R²=0.9351 r=0.9670 ρ=0.9476 τ=0.8204  Cliff=nan

            RAE     MAE      R2  Pearson  Spearman  Kendall  Cliff_acc
overall  0.2181  0.1985  0.9351   0.9670    0.9476   0.8204        NaN
active   1.5711  0.3295 -1.5465   0.3508    0.3558   0.2455        NaN


In [6]:
# Delta-ML correction for high-similarity test compounds
# Load delta model OOF if available
delta_te_path = DATA_PROCESSED / "te_oof_delta_ml.npy"
te_base = meta.predict(TE_stack)

# Compare individual models to ensemble on active compounds
print("\n=== Active compound (pEC50≥5.5) RAE per model ===")
for name, oof_m in zip(names, oofs):
    m = full_metrics(y_tr[active_mask], oof_m[active_mask])
    print(f"  {name:<35} RAE={m['RAE']:.4f}  Cliff_acc={m.get('Cliff_acc',float('nan')):.3f}")

te_preds = np.clip(te_base, y_tr.min()-0.5, y_tr.max()+0.5)
np.save(DATA_PROCESSED/"oof_creative_mega_ensemble.npy", oof)
np.save(DATA_PROCESSED/"te_oof_creative_mega_ensemble.npy", te_preds)
sub = pd.DataFrame({"Molecule Name":te["name"].values,"pEC50":te_preds})
assert len(sub)==513 and sub["pEC50"].notna().all()
p = SUBMISSIONS/"85_creative_mega_ensemble.csv"; sub.to_csv(p,index=False)
print(f"Saved {p}")
print(f"Test: min={te_preds.min():.2f} med={np.median(te_preds):.2f} max={te_preds.max():.2f}")



=== Active compound (pEC50≥5.5) RAE per model ===
  aux_features                        RAE=1.5873  Cliff_acc=nan
  bert_smiles                         RAE=5.2247  Cliff_acc=nan
  catboost                            RAE=3.2298  Cliff_acc=nan
  chemberta                           RAE=4.7394  Cliff_acc=nan
  chemberta_esm2_nr                   RAE=4.1503  Cliff_acc=nan
  chemberta_mtr                       RAE=4.1307  Cliff_acc=nan
  chemprop_aux                        RAE=3.5359  Cliff_acc=nan
  crossattn_chemberta_esm2            RAE=4.3307  Cliff_acc=nan
  crossattn_grover_esm2               RAE=4.3569  Cliff_acc=nan
  deep_ensemble                       RAE=3.6103  Cliff_acc=nan
  grand15                             RAE=3.5890  Cliff_acc=nan
  grand18                             RAE=3.4938  Cliff_acc=nan
  grand23                             RAE=3.5261  Cliff_acc=nan
  grand24                             RAE=3.5150  Cliff_acc=nan
  grand25                             RAE=3.5153  Cli